# FerrisRes — distillation smoke

**Not runnable until the blocked-config LoRA gradient is fixed.** The current
code panics with `lora_gradient_invalid_with_blocks` when a Block AttnRes
student has blocks and LoRA; this notebook is prepared so the backlog is
visible, and it stops loudly rather than training wrong.

Prerequisites: a `.deb` with that fix, the corpus as a Kaggle dataset, and a
Kaggle API token in Secrets for checkpoint upload.

In [ ]:
from pathlib import Path
    import os, subprocess

    DEB_DIR = Path("/kaggle/input/ferrisres-deb")
    CORPUS = Path("/kaggle/input/ferrisres-corpus/train.jsonl")
    TEACHER = Path("/kaggle/input/qwen3-0.6b/Qwen3-0.6B-Q8_0.gguf")
    TOKENIZER = Path("/kaggle/input/qwen3-0.6b/tokenizer.json")
    OUT = Path("/kaggle/working/distill"); OUT.mkdir(parents=True, exist_ok=True)
    RUN_LONG = False   # flip only after the smoke passes

In [ ]:
!nvidia-smi -L
    deb = sorted(DEB_DIR.glob("ferrisres_*_amd64.deb"))
    assert deb, "attach the .deb dataset"
    !dpkg -i {deb[-1]}
    !ferrisres devices

In [ ]:
smoke = [
        "ferrisres", "distill",
        "--model-path", str(TEACHER), "--model-format", "gguf",
        "--config", "qwen3-0.6b", "--tokenizer", str(TOKENIZER),
        "--data", str(CORPUS), "--data-format", "jsonl", "--chat-template", "qwen",
        "--require-provenance", "--seq-len", "128", "--steps", "10",
        "--checkpoint-every", "5", "--output", str(OUT / "smoke"),
    ]
    out = subprocess.run(smoke, text=True, capture_output=True)
    print(out.stdout[-2000:]); print(out.stderr[-2000:])
    if "lora_gradient_invalid_with_blocks" in out.stderr + out.stdout:
        raise SystemExit("STOP: the LoRA gradient path is not fixed in this build; "
                         "do not train. See docs/tasks/train-and-export.md.")
    assert out.returncode == 0, f"smoke failed: {out.returncode}"

In [ ]:
if RUN_LONG:
        # Resume from the smoke, checkpoint every 20 minutes, upload each time.
        # The command must carry --resume; a run without it restarts the schedule.
        cmd = [
            "ferrisres", "distill",
            "--model-path", str(TEACHER), "--model-format", "gguf",
            "--config", "qwen3-0.6b", "--tokenizer", str(TOKENIZER),
            "--data", str(CORPUS), "--data-format", "jsonl", "--chat-template", "qwen",
            "--require-provenance", "--seq-len", "512", "--steps", "2000",
            "--checkpoint-every", "200", "--output", str(OUT / "run"),
            "--resume", str(OUT / "smoke" / "latest"),
        ]
        !{cmd}
    else:
        print("RUN_LONG is False; smoke only.")

## After the run

`evaluate` and `benchmark` on a quiet machine, with the checkpoint hash. A
loss curve is not an evaluation; see `docs/eval-plan.md`.